In [1]:
import numpy as np

全体としては、まずフィルターの各位置について、入力データのうちそのマスが適用される部分を全部抜き取ってくる

その次に、フィルターの各位置について1番目を取り出すと一回目のフィルター適用でフィルターがかかる全部分がとってこれる
2番目以降も同様

2×2の3チャネルのフィルターを10個（重みはランダムで）生成

In [57]:
y=np.array([
    [
        [1,2],
        [3,4],
        [5,6]
    ],
    [
        [7,8],
        [9,10],
        [11,12]
    ]
])

In [58]:
print(y)
y.shape

[[[ 1  2]
  [ 3  4]
  [ 5  6]]

 [[ 7  8]
  [ 9 10]
  [11 12]]]


(2, 3, 2)

In [59]:
y.transpose(1,0,2)

array([[[ 1,  2],
        [ 7,  8]],

       [[ 3,  4],
        [ 9, 10]],

       [[ 5,  6],
        [11, 12]]])

In [12]:
x[0]

array([[[0.73508772, 0.37062985],
        [0.62108777, 0.23592719]],

       [[0.90434054, 0.73875196],
        [0.64675295, 0.57559321]],

       [[0.06587099, 0.72198475],
        [0.73649239, 0.94374536]]])

In [14]:
x[0,0]

array([[0.73508772, 0.37062985],
       [0.62108777, 0.23592719]])

あくまで入力データのみを変えることに注意。フィルター側はconvレイヤの中で明示的にバラす。

チャネル数1の一種類のフィルター

In [90]:
x=np.random.rand(1,2,4,4)

In [91]:
x

array([[[[0.8580499 , 0.15720481, 0.21712998, 0.06267282],
         [0.50966178, 0.85594947, 0.39792467, 0.35756191],
         [0.07183228, 0.27134525, 0.67540537, 0.44029177],
         [0.04630695, 0.09757902, 0.58149779, 0.29810425]],

        [[0.56086794, 0.67697015, 0.41648341, 0.15260967],
         [0.09999485, 0.28105169, 0.05691469, 0.79267551],
         [0.89411135, 0.95329045, 0.66561435, 0.74548931],
         [0.1867959 , 0.06591532, 0.37021135, 0.18260065]]]])

In [92]:
x_temp=np.random.rand(2,3,4)
x_temp

array([[[0.25346756, 0.15494571, 0.95153692, 0.98339243],
        [0.87490004, 0.96844472, 0.14254218, 0.31649923],
        [0.88787607, 0.85561887, 0.7726598 , 0.24652955]],

       [[0.40747806, 0.80115405, 0.3506548 , 0.78390776],
        [0.06637651, 0.31113559, 0.62880578, 0.65047376],
        [0.48044564, 0.38130062, 0.76546314, 0.0604289 ]]])

In [93]:
x_temp.transpose(1, 0, 2)

array([[[0.25346756, 0.15494571, 0.95153692, 0.98339243],
        [0.40747806, 0.80115405, 0.3506548 , 0.78390776]],

       [[0.87490004, 0.96844472, 0.14254218, 0.31649923],
        [0.06637651, 0.31113559, 0.62880578, 0.65047376]],

       [[0.88787607, 0.85561887, 0.7726598 , 0.24652955],
        [0.48044564, 0.38130062, 0.76546314, 0.0604289 ]]])

In [94]:
input_data=x.copy()
filter_h=2
filter_w=2
stride=1
pad=0

In [95]:
N,C,H,W=input_data.shape

# 出力の特徴マップの高さと横幅
out_h=(H+2*pad-filter_h) // stride +1
out_w=(W+2*pad-filter_w) // stride +1

print(N,C,H,W)
print(out_h,out_w)

1 2 4 4
3 3


In [96]:
# np.padで配列の端に値を足すことができる。constantの場合は0埋め。
# 引数については、N軸に前後0、C軸に前後0、H軸に前後padずつ（上下のpadに対応）、W軸に前後padずつ（左右のpadに対応）
img=np.pad(input_data, [(0,0), (0,0), (pad,pad),(pad,pad)], "constant")

In [97]:
img

array([[[[0.8580499 , 0.15720481, 0.21712998, 0.06267282],
         [0.50966178, 0.85594947, 0.39792467, 0.35756191],
         [0.07183228, 0.27134525, 0.67540537, 0.44029177],
         [0.04630695, 0.09757902, 0.58149779, 0.29810425]],

        [[0.56086794, 0.67697015, 0.41648341, 0.15260967],
         [0.09999485, 0.28105169, 0.05691469, 0.79267551],
         [0.89411135, 0.95329045, 0.66561435, 0.74548931],
         [0.1867959 , 0.06591532, 0.37021135, 0.18260065]]]])

In [98]:
# 一旦6次元のゼロ配列を作る？
col=np.zeros((N,C,filter_h,filter_w,out_h,out_w))

In [99]:
col.shape

(1, 2, 2, 2, 3, 3)

フィルターのサイズの各マス（今回は2×2）に、出力サイズのゼロ配列を作る

In [100]:
col

array([[[[[[0., 0., 0.],
           [0., 0., 0.],
           [0., 0., 0.]],

          [[0., 0., 0.],
           [0., 0., 0.],
           [0., 0., 0.]]],


         [[[0., 0., 0.],
           [0., 0., 0.],
           [0., 0., 0.]],

          [[0., 0., 0.],
           [0., 0., 0.],
           [0., 0., 0.]]]],



        [[[[0., 0., 0.],
           [0., 0., 0.],
           [0., 0., 0.]],

          [[0., 0., 0.],
           [0., 0., 0.],
           [0., 0., 0.]]],


         [[[0., 0., 0.],
           [0., 0., 0.],
           [0., 0., 0.]],

          [[0., 0., 0.],
           [0., 0., 0.],
           [0., 0., 0.]]]]]])

In [101]:
# フィルターの位置についてループする
# 例えば2×2なら左上、右上、左下、右下
for y in range(filter_h):
    
    # フィルター（のうち例えば左上のマス）をかける部分について、入力側の最初の1マス目はフィルターのマスを表す座標(0,0)と一致する。
    # 最後にかける部分は最初の部分からstride*out_h分動いた場所
    # これは、まず一回動くとフィルター位置はstride分ずれる。ずれ回数は、出力のhのうち最初の一回分である1を引いたout_h-1。
    # ただし、後ろのスライスで  a:bとしたときにbの手前までを取るので余分にstride*out_hとしている
    y_max=y+stride*out_h
    for x in range(filter_w):
        x_max=x+stride*out_w
        # 下記で、例えば入力データのうち、フィルターの左上のマスが通過する部分の要素を全部抜き出せる。
        # また、colの左上の3かけ3配列にその抜き出し結果を入れる
        col[:,:,y,x,:,:]=img[:,:,y:y_max:stride,x:x_max:stride]

In [102]:
col

array([[[[[[0.8580499 , 0.15720481, 0.21712998],
           [0.50966178, 0.85594947, 0.39792467],
           [0.07183228, 0.27134525, 0.67540537]],

          [[0.15720481, 0.21712998, 0.06267282],
           [0.85594947, 0.39792467, 0.35756191],
           [0.27134525, 0.67540537, 0.44029177]]],


         [[[0.50966178, 0.85594947, 0.39792467],
           [0.07183228, 0.27134525, 0.67540537],
           [0.04630695, 0.09757902, 0.58149779]],

          [[0.85594947, 0.39792467, 0.35756191],
           [0.27134525, 0.67540537, 0.44029177],
           [0.09757902, 0.58149779, 0.29810425]]]],



        [[[[0.56086794, 0.67697015, 0.41648341],
           [0.09999485, 0.28105169, 0.05691469],
           [0.89411135, 0.95329045, 0.66561435]],

          [[0.67697015, 0.41648341, 0.15260967],
           [0.28105169, 0.05691469, 0.79267551],
           [0.95329045, 0.66561435, 0.74548931]]],


         [[[0.09999485, 0.28105169, 0.05691469],
           [0.89411135, 0.95329045, 0.66561435],


(N, C, filter_h, filter_w, out_h, out_w)

上記から下記に並べ替え
(N, out_h, out_w, C, filter_h, filter_w)

結果的には

現在左上が見る成分の配列、右上が見る成分の配列、左下が見る成分の配列、右下が見る成分の配列の4つが並んでいるのを下記のようにする

4つの1番目の値を取ったもの（つまり一回めのフィルターで見る値一覧）
4つの2番目の値を取ったもの（つまり2回めのフィルターで見る値一覧）、、、

In [103]:
col=col.transpose(0,4,5,1,2,3)
print(col.shape)
col

(1, 3, 3, 2, 2, 2)


array([[[[[[0.8580499 , 0.15720481],
           [0.50966178, 0.85594947]],

          [[0.56086794, 0.67697015],
           [0.09999485, 0.28105169]]],


         [[[0.15720481, 0.21712998],
           [0.85594947, 0.39792467]],

          [[0.67697015, 0.41648341],
           [0.28105169, 0.05691469]]],


         [[[0.21712998, 0.06267282],
           [0.39792467, 0.35756191]],

          [[0.41648341, 0.15260967],
           [0.05691469, 0.79267551]]]],



        [[[[0.50966178, 0.85594947],
           [0.07183228, 0.27134525]],

          [[0.09999485, 0.28105169],
           [0.89411135, 0.95329045]]],


         [[[0.85594947, 0.39792467],
           [0.27134525, 0.67540537]],

          [[0.28105169, 0.05691469],
           [0.95329045, 0.66561435]]],


         [[[0.39792467, 0.35756191],
           [0.67540537, 0.44029177]],

          [[0.05691469, 0.79267551],
           [0.66561435, 0.74548931]]]],



        [[[[0.07183228, 0.27134525],
           [0.04630695, 0.09757902]

-1の部分はnumpy側が整合性とれるように調整してくれるやつ

引数が2な時点で2次元配列へのreshapeであり、各フィルター適用ごとに一本にしたいので、フィルターサイズごとに行にする。よって第一引数はフィルターの適用回数（チャネル複数ならその合計）

In [104]:
col=col.reshape(N*out_h*out_w,-1)

In [105]:
col

array([[0.8580499 , 0.15720481, 0.50966178, 0.85594947, 0.56086794,
        0.67697015, 0.09999485, 0.28105169],
       [0.15720481, 0.21712998, 0.85594947, 0.39792467, 0.67697015,
        0.41648341, 0.28105169, 0.05691469],
       [0.21712998, 0.06267282, 0.39792467, 0.35756191, 0.41648341,
        0.15260967, 0.05691469, 0.79267551],
       [0.50966178, 0.85594947, 0.07183228, 0.27134525, 0.09999485,
        0.28105169, 0.89411135, 0.95329045],
       [0.85594947, 0.39792467, 0.27134525, 0.67540537, 0.28105169,
        0.05691469, 0.95329045, 0.66561435],
       [0.39792467, 0.35756191, 0.67540537, 0.44029177, 0.05691469,
        0.79267551, 0.66561435, 0.74548931],
       [0.07183228, 0.27134525, 0.04630695, 0.09757902, 0.89411135,
        0.95329045, 0.1867959 , 0.06591532],
       [0.27134525, 0.67540537, 0.09757902, 0.58149779, 0.95329045,
        0.66561435, 0.06591532, 0.37021135],
       [0.67540537, 0.44029177, 0.58149779, 0.29810425, 0.66561435,
        0.74548931, 0.370211

In [110]:

def im2col(input_data,filter_h,filter_w,stride=1,pad=0):
    
    N,C,H,W=input_data.shape
    out_h=(H+2*pad-filter_h) // stride +1
    out_w=(W+2*pad-filter_w) // stride +1
    
    # np.padで配列の端に値を足すことができる。constantの場合は0埋め。
    # 引数については、N軸に前後0、C軸に前後0、H軸に前後padずつ（上下のpadに対応）、W軸に前後padずつ（左右のpadに対応）
    img=np.pad(input_data, [(0,0), (0,0), (pad,pad),(pad,pad)], "constant")
    
    # 一旦6次元配列を作る？
    col=np.zeros((N,C,filter_h,filter_w,out_h,out_w))
    
    # フィルターの位置についてループする
    # 例えば2×2なら左上、右上、左下、右下
    for y in range(filter_h):
        y_max=y+stride*out_h
        for x in range(filter_w):
            x_max=x+stride*out_w
            col[:,:,y,x,:,:]=img[:,:,y:y_max:stride,x:x_max:stride]
            
    col=col.transpose(0,4,5,1,2,3).reshape(N*out_h*out_w,-1)
    
    return col

In [112]:
x1=np.random.rand(1,3,7,7)
print(x1)
col1=im2col(x1,5,5,stride=1,pad=0)
print(col1)
print(col1.shape)


[[[[0.38113264 0.24708298 0.60605144 0.58124872 0.57128874 0.45134569
    0.31727626]
   [0.53481588 0.90590884 0.05157667 0.76740658 0.80310795 0.11391715
    0.39075528]
   [0.20781889 0.98009361 0.86119508 0.63251402 0.29101085 0.44890138
    0.65899811]
   [0.77749469 0.44209435 0.8612256  0.73519778 0.85171468 0.9948067
    0.15177164]
   [0.34124949 0.82611238 0.83337149 0.06066522 0.02109066 0.90807117
    0.05202358]
   [0.76114924 0.8774667  0.21723888 0.59643951 0.78897377 0.09094225
    0.32798954]
   [0.57874247 0.25861033 0.60360445 0.93553412 0.28074436 0.26632914
    0.0877903 ]]

  [[0.6422284  0.71313182 0.93515756 0.72410322 0.07598793 0.17732245
    0.45875462]
   [0.06086207 0.08005412 0.91271815 0.58122397 0.84238134 0.14361467
    0.08696588]
   [0.72483719 0.87588944 0.33443052 0.23163749 0.56498676 0.25746773
    0.21845672]
   [0.41637431 0.24780269 0.71961292 0.53685423 0.29213842 0.26791894
    0.90157725]
   [0.67749446 0.40341012 0.77566201 0.40474152 0.863

In [113]:
x2=np.random.rand(10,3,7,7)
print(x2)
col2=im2col(x2,5,5,stride=1,pad=0)
print(col2)
print(col2.shape)

[[[[0.32892222 0.0413568  0.49533635 ... 0.29611401 0.20948577
    0.80633916]
   [0.06275504 0.65757493 0.68772082 ... 0.80804911 0.37979702
    0.28302295]
   [0.70911977 0.16323558 0.65609476 ... 0.34902075 0.28773239
    0.51394543]
   ...
   [0.07447056 0.4913586  0.31626406 ... 0.91012206 0.65764251
    0.11915403]
   [0.85463017 0.00503356 0.52516282 ... 0.77991514 0.00200062
    0.22827294]
   [0.62962744 0.20807995 0.2179084  ... 0.44142611 0.78590237
    0.01085923]]

  [[0.03482142 0.20619507 0.08124448 ... 0.89928067 0.97009678
    0.75377327]
   [0.47846339 0.44982524 0.68352565 ... 0.68129943 0.95927323
    0.37652475]
   [0.03572111 0.73239877 0.65365396 ... 0.85216527 0.74214655
    0.27446117]
   ...
   [0.48908854 0.93340835 0.46897586 ... 0.71098005 0.03444988
    0.71931816]
   [0.71172891 0.95974386 0.25268067 ... 0.56278388 0.44959103
    0.29366933]
   [0.01977501 0.89938152 0.46293259 ... 0.53206946 0.7438449
    0.60547194]]

  [[0.00645848 0.47388251 0.5113604